# DATA WIZARDS — Reproducible data-quality audit

Run **Kernel → Restart & Run All** from the repository root (or set `RAW_DIR`). Source files are never overwritten. Outputs are `docs/dq_log.csv`, `docs/dq_decisions.csv`, `docs/dq_rejects.csv`, `docs/dq_review.csv`, and `docs/dq_category_variants.csv`.

**Scope:** 1 Jan 2024–30 Jun 2025 applies to **order dates only**. Flag (rather than guess repairs to) contradictory or malformed dates. Distinguish diagnostics from approved cleaning decisions. All counts here are computed from raw files; previously quoted counts are not hardcoded.

**Note:** This is a diagnostic audit, not a production cleaning pipeline. Ambiguous IDs and unrecognized categories remain flagged for approval; cleaned tables must be produced by `include/clean/` after decisions are finalized.


In [1]:
from pathlib import Path
import os, re
import pandas as pd
import numpy as np
from IPython.display import display

# Resolve paths correctly whether the notebook is launched from project root or notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / 'data' / 'raw').is_dir() and (PROJECT_ROOT.parent / 'data' / 'raw').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = Path(os.environ.get('RAW_DIR', str(PROJECT_ROOT / 'data' / 'raw')))
DOCS_DIR = Path(os.environ.get('DOCS_DIR', str(PROJECT_ROOT / 'docs')))
DOCS_DIR.mkdir(parents=True, exist_ok=True)
if not RAW_DIR.is_dir():
    raise FileNotFoundError(f"RAW_DIR={RAW_DIR.resolve()} not found; put the nine CSVs there or set RAW_DIR")

EXPECTED = ('customers','locations','orders','order_items','payments','products','returns','sellers','shipping')
ALIASES = {'orderitems':'order_items', 'order_details':'order_items', 'orderdetails':'order_items',
           'customer':'customers','city':'locations','cities':'locations','location':'locations','locations':'locations','order':'orders','payment':'payments',
           'product':'products','return':'returns','seller':'sellers','shipment':'shipping','shipments':'shipping'}

def file_key(path):
    stem = re.sub(r'[^a-z0-9]+', '_', path.stem.lower()).strip('_')
    for key in EXPECTED:
        if stem == key or stem.endswith('_' + key) or stem.startswith(key + '_'):
            return key
    return ALIASES.get(stem)

paths = sorted(RAW_DIR.glob('*.csv'))
data = {}
source_paths = {}
for path in paths:
    key = file_key(path)
    if key is None:
        print(f'Unrecognized CSV (not loaded): {path.name}')
        continue
    if key in data:
        raise ValueError(f'Duplicate source for {key}: {source_paths[key]} / {path}')
    data[key] = pd.read_csv(path, low_memory=False, dtype='string')
    source_paths[key] = path.name
missing_files = set(EXPECTED) - set(data)
if missing_files:
    raise FileNotFoundError(f'Missing expected CSVs: {sorted(missing_files)}. Found: {list(source_paths.values())}')
print('Loaded:', {k:(v.shape,source_paths[k]) for k,v in data.items()})

log_rows, reject_parts, review_parts, decisions = [], [], [], []
def log(file, check, count, details=''):
    log_rows.append({'file':file, 'check':check, 'count':int(count), 'details':details})
def flag(file, mask, reason, *, reject=False, details=''):
    mask = pd.Series(mask, index=data[file].index).fillna(False).astype(bool)
    log(file, reason, mask.sum(), details)
    if mask.any():
        rows = data[file].loc[mask].copy()
        rows.insert(0, 'source_row', rows.index + 2)  # header is row 1
        rows.insert(0, 'source_file', file)
        rows.insert(0, 'reason_code', reason)
        (reject_parts if reject else review_parts).append(rows)
    return mask

def decide(file, finding, treatment, rationale):
    decisions.append({'file':file,'finding':finding,'treatment':treatment,'rationale':rationale})

def money(s):
    return (s.astype('string').str.replace(r'(?i)\b(?:rs\.?|pkr)\s*', '',regex=True)
            .str.replace(',', '',regex=False).str.strip())

def numeric_check(file, col, *, currency=False):
    if col not in data[file]:
        return pd.Series(np.nan,index=data[file].index,dtype='float64')
    raw = data[file][col]
    cleaned = money(raw) if currency else raw.astype('string').str.strip().str.replace(',','',regex=False)
    out = pd.to_numeric(cleaned, errors='coerce')
    flag(file, raw.notna() & out.isna(), f'INVALID_NUMERIC_{col.upper()}', reject=True,
         details='Value became null on numeric parsing')
    return out

date_formats_observed = {}
def parsed_date(file, col):
    if col not in data[file]:
        return None
    raw=data[file][col]
    # Parse ISO first: dayfirst=True + format='mixed' incorrectly flips ISO month/day.
    raw_text=raw.astype('string').str.strip()
    patterns=raw_text.dropna().str.replace(r'\d','9',regex=True).value_counts()
    date_formats_observed[(file,col)]=patterns
    dt=pd.to_datetime(raw_text,errors='coerce',format='%Y-%m-%d')
    for fmt in ('%d/%m/%Y','%b %d %Y','%B %d %Y','%Y/%m/%d','%Y-%m-%d %H:%M:%S', '%d/%m/%Y %H:%M:%S'):
        dt=dt.fillna(pd.to_datetime(raw_text,errors='coerce',format=fmt))
    flag(file,raw.notna() & dt.isna(),f'INVALID_DATE_{col.upper()}',reject=(file not in ('customers','sellers')))
    log(file,f'MISSING_{col.upper()}',raw.isna().sum())
    return dt


Loaded: {'customers': ((45000, 8), 'customers.csv'), 'locations': ((650, 7), 'locations.csv'), 'order_items': ((315143, 7), 'order_items.csv'), 'orders': ((163200, 8), 'orders.csv'), 'payments': ((168111, 6), 'payments.csv'), 'products': ((3200, 9), 'products.csv'), 'returns': ((16048, 6), 'returns.csv'), 'sellers': ((420, 6), 'sellers.csv'), 'shipping': ((151228, 7), 'shipping.csv')}


## 1. Audit source files, IDs and exact duplicates

In [2]:
PRIMARY_KEYS = {'customers':'customer_id','locations':'city_id','orders':'order_id',
                'order_items':'order_item_id','payments':'payment_id','products':'product_id',
                'returns':'return_id','sellers':'seller_id','shipping':'shipment_id'}
clean = {k:v.copy() for k,v in data.items()}
for file, df in data.items():
    log(file,'ROWS_RAW',len(df))
    log(file,'COLUMNS',len(df.columns))
    for col in df.columns:
        log(file,f'NULL_{col}',df[col].isna().sum())
    exact = df.duplicated(keep='first')
    flag(file,exact,'EXACT_DUPLICATE',reject=True)
    clean[file] = df.loc[~exact].copy()
    pk=PRIMARY_KEYS[file]
    if pk in df:
        flag(file,df[pk].notna() & df[pk].duplicated(keep=False),f'DUPLICATE_KEY_{pk.upper()}')
        flag(file,df[pk].isna(),f'MISSING_KEY_{pk.upper()}',reject=True)
    log(file,'ROWS_AFTER_EXACT_DEDUP',len(clean[file]))
decide('orders/payments','Exact identical records','Drop later identical copy; export reject row',
       'All rate denominators use exact-deduplicated orders; repeat payment IDs are checked separately')
display(pd.DataFrame(log_rows).query("check in ['ROWS_RAW','EXACT_DUPLICATE','ROWS_AFTER_EXACT_DEDUP']"))

,file,check,count,details
0,customers,ROWS_RAW,45000,
10,customers,EXACT_DUPLICATE,0,
13,customers,ROWS_AFTER_EXACT_DEDUP,45000,
14,locations,ROWS_RAW,650,
23,locations,EXACT_DUPLICATE,0,
26,locations,ROWS_AFTER_EXACT_DEDUP,650,
27,order_items,ROWS_RAW,315143,
36,order_items,EXACT_DUPLICATE,0,
39,order_items,ROWS_AFTER_EXACT_DEDUP,315143,
40,orders,ROWS_RAW,163200,


## 2. Dates: order window and chronological checks (no speculative repairs)

In [3]:
date_cache={}
for file, df in data.items():
    for col in df.columns:
        if 'date' in col.lower() or col.lower().endswith('_at'):
            dt=parsed_date(file,col)
            if dt is not None:date_cache[(file,col)]=dt

order_date_column = next((c for c in ('order_date','ordered_at','created_at') if c in data['orders']),None)
if order_date_column is None:
    raise KeyError('No order date column found in orders; configure order_date_column')
order_dates=date_cache.get(('orders',order_date_column))
flag('orders', order_dates.notna() & ((order_dates<pd.Timestamp('2024-01-01')) |
     (order_dates>=pd.Timestamp('2025-07-01'))), 'ORDER_DATE_OUTSIDE_WINDOW',reject=True)

# Compare event dates against linked order dates, including events legitimately in July 2025.
order_date_by_id = pd.Series(order_dates.values,index=data['orders']['order_id']).dropna().groupby(level=0).first()
for file in ('payments','shipping','returns'):
    if 'order_id' not in data[file]:continue
    od=data[file]['order_id'].map(order_date_by_id)
    for col in data[file]:
        if ('date' not in col.lower() and not col.lower().endswith('_at')):continue
        dt=date_cache.get((file,col))
        if dt is not None:
            flag(file, dt.notna() & od.notna() & (dt < od), f'{col.upper()}_BEFORE_ORDER')

# Within shipping, use actual column names if present.
ship=data['shipping']
def first_col(df, names):return next((c for c in names if c in df.columns),None)
shipped=first_col(ship,['shipped_date','ship_date','shipped_at'])
delivered=first_col(ship,['delivered_date','delivery_date','delivered_at'])
if shipped and delivered:
    a=date_cache.get(('shipping',shipped));b=date_cache.get(('shipping',delivered))
    flag('shipping',a.notna() & b.notna() & (b<a),'DELIVERED_BEFORE_SHIPPED')
decide('orders','Order dates outside window or unparseable','Quarantine pending verified source correction',
       'Never infer a date solely from adjacent order IDs')
decide('customers/sellers','Malformed registration or join dates','Null parsed date, keep row; retain flagged source',
       'Rejecting parent entities would create artificial foreign-key orphans')
# Persist format profiles to diagnose new formats without guessing date order.
date_format_rows=[]
for (file,col),counts in date_formats_observed.items():
    for pattern,count in counts.items():
        date_format_rows.append({'file':file,'column':col,'pattern':pattern,'count':int(count)})
date_format_df=pd.DataFrame(date_format_rows,columns=['file','column','pattern','count'])
display(date_format_df.head(30))


,file,column,pattern,count
0,customers,registration_date,9999-99-99,38087
1,customers,registration_date,99/99/9999,3760
2,customers,registration_date,Jan 99 9999,299
3,customers,registration_date,Feb 99 9999,260
4,customers,registration_date,Dec 99 9999,257
5,customers,registration_date,Mar 99 9999,228
6,customers,registration_date,Apr 99 9999,212
7,customers,registration_date,May 99 9999,192
8,customers,registration_date,Jun 99 9999,175
9,customers,registration_date,Nov 99 9999,162


## 3. Foreign keys and orphan handling

In [4]:
RELATIONSHIPS=[('orders','customer_id','customers','customer_id'),
 ('orders','ship_city_id','locations','city_id'),('order_items','order_id','orders','order_id'),
 ('order_items','product_id','products','product_id'),('order_items','seller_id','sellers','seller_id'),
 ('payments','order_id','orders','order_id'),('products','seller_id','sellers','seller_id'),
 ('returns','order_item_id','order_items','order_item_id'),('sellers','city_id','locations','city_id'),
 ('shipping','order_id','orders','order_id')]
for child,fk,parent,pk in RELATIONSHIPS:
    if fk not in data[child] or pk not in data[parent]:continue
    mask=data[child][fk].notna() & ~data[child][fk].isin(data[parent][pk])
    flag(child,mask,f'ORPHAN_{fk.upper()}_TO_{parent.upper()}',reject=(child=='returns'))
    flag(child,data[child][fk].isna(),f'MISSING_FK_{fk.upper()}')
flag('orders',data['orders']['ship_city_id'].isna(), 'UNKNOWN_SHIP_CITY') if 'ship_city_id' in data['orders'] else None
decide('orders','Unknown customer or shipping city','Unknown dimension member (distinct from actual customer city)',
       'Preserves order counts and revenue; keep original FK and emit a modeled Unknown key downstream')
decide('order_items','Missing product dimension','Unknown product member; preserve financial line',
       'Do not lose sales by dropping financial facts')
decide('returns','Unknown order item FK','Quarantine from return-rate numerator, export for review',
       'Cannot attribute a return to a qualifying sold line')

## 4. Categorical variants — profile *every* low-cardinality string column

In [5]:
variant_rows=[]
for file,df in data.items():
    for col in df.columns:
        s=df[col]
        if not (pd.api.types.is_string_dtype(s.dtype) or s.dtype==object):continue
        distinct=s.dropna().nunique()
        if distinct>max(100, int(.05*len(df))):continue
        norm=s.astype('string').str.strip().str.casefold().str.replace(r'\s+',' ',regex=True)
        groups=pd.DataFrame({'raw':s,'normalized':norm}).dropna().drop_duplicates()
        for _,g in groups.groupby('normalized',dropna=False):
            variants=g['raw'].astype(str).tolist()
            if len(variants)>1:
                variant_rows.append({'file':file,'column':col,'normalized':g['normalized'].iloc[0],
                                     'variants':' | '.join(variants),'variant_count':len(variants)})
        log(file,f'CATEGORY_DISTINCT_{col}',distinct)
variants_df=pd.DataFrame(variant_rows,columns=['file','column','normalized','variants','variant_count'])
display(variants_df.head(40))

# Mappings derived from observed source values; retain raw columns for auditability.
def mapped(groups):
    return {variant.casefold():canonical for canonical, variants in groups.items() for variant in variants}
CATEGORY_MAP=mapped({
 'Electronics':['Electronics','Elec.','Electronic Items'],
 'Fashion':['Fashion','Fashion & Apparel','Clothing'],
 'Mobile & Accessories':['Mobile & Accessories','Mobile Acc.','Mobile and Accessories','Mobiles'],
 'Home & Kitchen':['Home & Kitchen','Home and Kitchen','Home&Kitchen'],
 'Beauty & Personal Care':['Beauty & Personal Care','Beauty and Personal Care','Beauty','Personal Care'],
 'Footwear':['Footwear','Shoes'],
 'Grocery':['Grocery','Food & Grocery','Groceries'],
 'Sports & Outdoors':['Sports & Outdoors','Sports and Outdoors','Sports'],
 'Health & Wellness':['Health & Wellness','Health and Wellness','Health'],
 'Toys & Games':['Toys & Games','Toys and Games','Toys'],
 'Books & Stationery':['Books & Stationery','Books and Stationery','Books','Stationary'],
 'Automotive':['Automotive','Auto','Car Accessories']})
ORDER_STATUS_MAP=mapped({'Delivered':['Delivered','DELIVERD','Completed','Complete'],
 'Cancelled':['Cancelled','Canceled'],'Returned':['Returned','Return'],
 'In Transit':['In Transit','In-Transit'],'Shipped':['Shipped']})
PAYMENT_STATUS_MAP=mapped({'Paid':['Paid','Complete','Success'],
 'Cancelled':['Cancelled','Canceled'],'Refunded':['Refunded','Refund'],'Pending':['Pending']})
DELIVERY_STATUS_MAP=mapped({'Delivered':['Delivered','DELIVERD','Completed','Complete'],
 'Returned to Seller':['Returned to Seller','Return to Seller','RTO'],
 'In Transit':['In Transit','In-Transit'],'Shipped':['Shipped']})
RETURN_STATUS_MAP=mapped({'Refunded':['Refunded','Refund'],'Approved':['Approved'],
 'Rejected':['Rejected','Declined']})
CHANNEL_MAP=mapped({'App':['App'],'Web':['Web','Website'],
 'Mobile Web':['Mobile Web','MWeb'],'Mobile App':['Mobile App']})
PAYMENT_METHOD_MAP=mapped({'COD':['COD','Cash on Delivery'],
 'Card':['Card','Credit/Debit Card','Debit Card'],
 'JazzCash':['JazzCash','Jazz Cash'],'EasyPaisa':['EasyPaisa','Easy Paisa'],
 'Bank Transfer':['Bank Transfer','Bank Trf']})
COURIER_MAP=mapped({'BlueEx':['Blue Ex','Blue-Ex','BlueEx'],
 'Leopards':['Leopards','Leopard Courier'],'M&P':['M&P','M & P','MNP'],
 'PostEx':['PostEx'],'TCS':['TCS','T.C.S','TCS Express'],'Trax':['Trax']})
PROVINCE_MAP=mapped({'Punjab':['Punjab','Panjab'],'Sindh':['Sindh','Sind'],
 'Balochistan':['Balochistan','Baluchistan'],
 'Khyber Pakhtunkhwa':['Khyber Pakhtunkhwa','KP','KPK'],
 'Gilgit-Baltistan':['Gilgit Baltistan','Gilgit-Baltistan','GB'],
 'Azad Jammu & Kashmir':['Azad Jammu & Kashmir','Azad Kashmir'],
 'Islamabad':['Islamabad']})
VOCABS={'category':CATEGORY_MAP,'order_status':ORDER_STATUS_MAP,
 'payment_status':PAYMENT_STATUS_MAP,'delivery_status':DELIVERY_STATUS_MAP,
 'return_status':RETURN_STATUS_MAP,'channel':CHANNEL_MAP,
 'payment_method':PAYMENT_METHOD_MAP,'courier':COURIER_MAP,'province':PROVINCE_MAP}
def map_vocab(series, mapping):
    raw=series.astype('string').str.strip().str.casefold().str.replace(r'\s+',' ',regex=True)
    return raw.map(mapping).fillna(series.astype('string').str.strip())
for file in clean:
    for col in clean[file].columns:
        if col in VOCABS:
            clean[file][col+'_standardized']=map_vocab(clean[file][col],VOCABS[col])
            unknown=clean[file][col].notna() & ~clean[file][col].astype('string').str.strip().str.casefold().str.replace(r'\s+',' ',regex=True).isin(VOCABS[col])
            log(file,f'UNMAPPED_{col.upper()}',unknown.sum())
        elif col in ('discount_code','gender','seller_tier','sub_category','brand','region','population_tier'):
            clean[file][col+'_normalized']=clean[file][col].astype('string').str.strip().str.upper()
# The standardized columns are provisional; approval belongs in include/clean/mappings.py.
decide('returns','Declined status','Rejected','Do not include rejected returns in return-rate numerator')
decide('shipping','Shipped vs In Transit','Keep distinct','In Transit counts toward NMV eligibility only if order-level business definition includes it; never mark delivered')

,file,column,normalized,variants,variant_count
0,customers,gender,f,F | f,2
1,customers,gender,female,female | Female,2
2,customers,gender,m,m | M,2
3,customers,gender,male,Male | male,2
4,locations,province,balochistan,Balochistan | BALOCHISTAN,2
5,locations,province,khyber pakhtunkhwa,Khyber Pakhtunkhwa | Khyber Pakhtunkhwa,2
6,locations,province,punjab,Punjab | punjab | PUNJAB,3
7,locations,province,sindh,Sindh | sindh | SINDH,3
8,orders,order_status,cancelled,Cancelled | cancelled | Cancelled | CANCELLED,4
9,orders,order_status,delivered,Delivered | DELIVERED | delivered,3


## 5. Numeric validation, price/weight/rating anomalies and discounted order totals

In [6]:
numbers={}
NUMERIC_COLS={'customers':['birth_year'],'locations':['latitude','longitude'],
 'orders':['order_total'],'order_items':['quantity','unit_price','discount_amount'],
 'payments':['amount'],'products':['list_price','cost_price','weight_kg'],
 'returns':['refund_amount'],'sellers':['rating']}
for file,cols in NUMERIC_COLS.items():
    for col in cols:
        if col in data[file]:numbers[(file,col)]=numeric_check(file,col,currency=col in ('unit_price','order_total','amount','list_price','cost_price','refund_amount','discount_amount'))

for file,col,cond,reason in [
 ('products','weight_kg',lambda x:x<=0,'NONPOSITIVE_WEIGHT'),
 ('products','cost_price',lambda x:x<0,'NEGATIVE_COST'),
 ('products','list_price',lambda x:x<=0,'NONPOSITIVE_LIST_PRICE'),
 ('sellers','rating',lambda x:(x<0)|(x>5),'RATING_OUTSIDE_0_5'),
 ('order_items','quantity',lambda x:x<=0,'NONPOSITIVE_QUANTITY')]:
    if (file,col) in numbers:flag(file,cond(numbers[(file,col)]).fillna(False),reason)
if all(('products',x) in numbers for x in ('cost_price','list_price')):
    flag('products',numbers['products','cost_price']>numbers['products','list_price'],'COST_ABOVE_LIST_PRICE')

# Only exact repeated orders are deduplicated; merge with m:1 validation.
orders=clean['orders'].copy()
items=clean['order_items'].copy()
order_codes=orders[['order_id','discount_code']].copy() if 'discount_code' in orders else orders[['order_id']].assign(discount_code=pd.NA)
order_codes['code']=order_codes['discount_code'].astype('string').str.strip().str.upper()
flag('orders',orders['order_id'].duplicated(keep=False),'DUPLICATE_ORDER_ID_AFTER_EXACT_DEDUP')
# Quarantine ambiguous IDs for reconciliation; do not select an arbitrary row.
ambiguous_ids=set(orders.loc[orders['order_id'].duplicated(keep=False),'order_id'].dropna())
if ambiguous_ids:
    log('orders','AMBIGUOUS_ORDER_ID_COUNT',len(ambiguous_ids),'Excluded from discount join and total check')
    flag('orders',data['orders']['order_id'].isin(ambiguous_ids),'AMBIGUOUS_ORDER_ID',reject=True)
    decide('orders','Conflicting rows share order_id','Quarantine affected orders from reconciliation',
           'Do not arbitrarily select one discount code or order total')
order_codes=order_codes.loc[~order_codes['order_id'].isin(ambiguous_ids)].copy()

items=items.merge(order_codes[['order_id','code']],on='order_id',how='left',validate='m:1')
for col in ('quantity','unit_price','discount_amount'):
    items[col]=pd.to_numeric(money(items[col]) if col!='quantity' else items[col],errors='coerce').astype('float64')
items['gross_amount']=items['quantity']*items['unit_price']
# Only percentage codes verified by the business may be included here.
# Derive percentage only for supported SAVE<number> codes (e.g., SAVE10, SAVE 15).
# Rates outside 0..100 and all other codes require review rather than silent guesses.
code_pct=pd.to_numeric(items['code'].str.extract(r'^SAVE\s*(\d{1,3})$',expand=False),errors='coerce')
items['code_rate']=(code_pct/100).where(code_pct.between(0,100)).astype('float64')
flag('orders',orders['discount_code'].notna() & ~orders['discount_code'].astype('string').str.strip().str.upper().str.fullmatch(r'SAVE\s*\d{1,3}').fillna(False),
     'UNRECOGNIZED_DISCOUNT_CODE')
items['discount_original_missing']=items['discount_amount'].isna()
items['discount_imputed']=False
no_code=items['discount_original_missing'] & (items['code'].isna() | items['code'].eq(''))
known_code=items['discount_original_missing'] & items['code_rate'].notna() & items['gross_amount'].notna()
items.loc[no_code,'discount_amount']=0.0
items.loc[known_code,'discount_amount']=(items.loc[known_code,'gross_amount']*items.loc[known_code,'code_rate']).round(2).astype('float64')
items.loc[no_code|known_code,'discount_imputed']=True
unresolved=items['discount_amount'].isna()
log('order_items','DISCOUNT_IMPUTED',items['discount_imputed'].sum())
log('order_items','DISCOUNT_UNRESOLVED',unresolved.sum(), 'Unknown code or invalid gross; never treat as zero')
log('order_items','ZERO_OR_MISSING_GROSS',(items['gross_amount'].isna()|(items['gross_amount']<=0)).sum())
log('order_items','NEGATIVE_DISCOUNT',(items['discount_amount']<0).sum())
log('order_items','DISCOUNT_ABOVE_GROSS',(items['discount_amount']>items['gross_amount']).sum())
items['line_total']=items['gross_amount']-items['discount_amount']
# min_count=1 keeps all-null orders null, instead of wrongly reporting 0.
grouped=items.groupby('order_id',dropna=False)['line_total']
calculated=grouped.sum(min_count=1)
# Reject partially-known totals rather than silently summing only known lines.
calculated=calculated.mask(grouped.count()<grouped.size())
calculated=calculated.mask(calculated.index.isin(ambiguous_ids))
orders['calculated_total']=orders['order_id'].map(calculated)
orders['order_total_num']=pd.to_numeric(money(orders['order_total']),errors='coerce').astype('float64')
orders['difference']=orders['order_total_num']-orders['calculated_total']
flag('orders',orders['difference'].abs()>0.01,'ORDER_TOTAL_MISMATCH_AFTER_DISCOUNT_REPAIR')
log('orders','TOTAL_UNCHECKABLE',orders['calculated_total'].isna().sum())
log('orders','TOTAL_EXCLUDED_AMBIGUOUS_ORDER_ID',len(ambiguous_ids))
flag('order_items',data['order_items']['discount_amount'].isna(),'DISCOUNT_MISSING_IN_SOURCE')
decide('order_items','Null discounts','Zero only if code missing; compute gross × verified code rate otherwise',
       'Unrecognized codes remain unresolved and require approved mapping')
display(orders.loc[orders['difference'].abs()>0.01,['order_id','order_total','calculated_total','difference']].head(20))

,order_id,order_total,calculated_total,difference
16,O0042918,1476,1475.60,0.40
41,O0130009,3080,3079.55,0.45
49,O0109828,3787,3786.80,0.20
55,O0114697,4595,2991.00,1604.00
57,O0116912,11270,11270.50,-0.50
61,O0076349,18448,18447.80,0.20
123,O0154003,4257,4256.80,0.20
137,O0049051,2421,2420.80,0.20
146,O0120349,9999999,4641.00,9995358.00
154,O0019950,3494,3493.65,0.35


## 6. Duplicate candidates — review, do not automatically merge

In [7]:
customers=data['customers'].copy()
for col in ('full_name','email','phone'):
    if col in customers:customers['norm_'+col]=customers[col].astype('string').str.strip().str.casefold()
if 'phone' in customers:customers['norm_phone']=customers['phone'].astype('string').str.replace(r'\D','',regex=True)
keys=[x for x in ('norm_full_name','norm_email','norm_phone','birth_year') if x in customers]
if len(keys)==4:
    complete=customers[keys].notna().all(axis=1) & customers['norm_email'].ne('') & customers['norm_phone'].ne('')
    flag('customers',complete & customers.duplicated(keys,keep=False),'POSSIBLE_CUSTOMER_DUPLICATE_4_FIELDS')
    for subset in (['norm_phone','norm_full_name'],['norm_email','birth_year']):
        mask=customers[subset].notna().all(axis=1) & customers.duplicated(subset,keep=False)
        flag('customers',mask,f'REVIEW_CUSTOMER_DUPLICATE_{"_".join(subset).upper()}')

products=data['products'].copy()
if all(c in products for c in ('product_name','cost_price','weight_kg')):
    products['norm_name']=(products['product_name'].astype('string').str.strip().str.casefold()
        .str.replace(r'\s*(?:-\s*new|\(2024\)|original)\s*$', '',regex=True).str.strip())
    products['norm_cost']=pd.to_numeric(money(products['cost_price']),errors='coerce').round(2)
    products['norm_weight']=pd.to_numeric(products['weight_kg'],errors='coerce').round(5)
    keys=['norm_name','norm_cost','norm_weight']
    mask=products[keys].notna().all(axis=1) & products.duplicated(keys,keep=False)
    flag('products',mask,'POSSIBLE_PRODUCT_DUPLICATE_NAME_COST_WEIGHT')
decide('customers/products','Near-duplicate entities','Investigate matched IDs, keep originals until crosswalk approved',
       'Missing keys do not count as evidence of equality; never rely on brand alone')

## 7. Payments: distinguish exact repeats, multiple attempts, and ambiguous methods

In [8]:
payments=clean['payments'].copy()
if 'order_id' in payments:
    counts=payments.groupby('order_id',dropna=False).size()
    multiple_ids=counts[counts>1].index
    flag('payments',payments['order_id'].isin(multiple_ids),'MULTIPLE_PAYMENT_ROWS_SAME_ORDER')
    method_col=first_col(payments,['payment_method','method'])
    if method_col:
        distinct_methods=payments.assign(method=payments[method_col].astype('string').str.strip().str.upper()).groupby('order_id')['method'].nunique(dropna=True)
        ambiguous=distinct_methods[distinct_methods>1].index
        flag('payments',payments['order_id'].isin(ambiguous),'MULTIPLE_METHODS_SAME_ORDER')
        # This is a diagnostic. Downstream must choose confirmed captured/settled method using payment status and timestamps.
        decide('payments','Multiple payment attempts or methods per order',
               'Use final successful captured payment if determinable; otherwise Unknown and review',
               'Do not arbitrarily choose first payment_id or sum duplicate attempts')


## 8. Export persistent diagnostics and cleaning decisions

In [9]:
dq_log=pd.DataFrame(log_rows,columns=['file','check','count','details'])
dq_decisions=pd.DataFrame(decisions,columns=['file','finding','treatment','rationale'])
def concat_flags(parts):
    # Different files have different schemas: schema-aligned CSV with metadata first.
    return pd.concat(parts,ignore_index=True,sort=False) if parts else pd.DataFrame(columns=['reason_code','source_file','source_row'])
dq_rejects=concat_flags(reject_parts)
dq_review=concat_flags(review_parts)
dq_log.to_csv(DOCS_DIR/'dq_log.csv',index=False)
dq_decisions.to_csv(DOCS_DIR/'dq_decisions.csv',index=False)
dq_rejects.to_csv(DOCS_DIR/'dq_rejects.csv',index=False)
dq_review.to_csv(DOCS_DIR/'dq_review.csv',index=False)
variants_df.to_csv(DOCS_DIR/'dq_category_variants.csv',index=False)
date_format_df.to_csv(DOCS_DIR/'dq_date_formats.csv',index=False)
print('Wrote:',*[str(p) for p in DOCS_DIR.glob('dq_*.csv')],sep='\n- ')
display(dq_log.tail(25))


Wrote:
- C:\Users\Syscom\Desktop\nexusmart-codeconnect\docs\dq_category_variants.csv
- C:\Users\Syscom\Desktop\nexusmart-codeconnect\docs\dq_date_formats.csv
- C:\Users\Syscom\Desktop\nexusmart-codeconnect\docs\dq_decisions.csv
- C:\Users\Syscom\Desktop\nexusmart-codeconnect\docs\dq_log.csv
- C:\Users\Syscom\Desktop\nexusmart-codeconnect\docs\dq_rejects.csv
- C:\Users\Syscom\Desktop\nexusmart-codeconnect\docs\dq_review.csv


,file,check,count,details
208,returns,INVALID_NUMERIC_REFUND_AMOUNT,0,Value became null on numeric parsing
209,sellers,INVALID_NUMERIC_RATING,0,Value became null on numeric parsing
210,products,NONPOSITIVE_WEIGHT,10,
211,products,NEGATIVE_COST,0,
212,products,NONPOSITIVE_LIST_PRICE,8,
213,sellers,RATING_OUTSIDE_0_5,11,
214,order_items,NONPOSITIVE_QUANTITY,307,
215,products,COST_ABOVE_LIST_PRICE,75,
216,orders,DUPLICATE_ORDER_ID_AFTER_EXACT_DEDUP,0,
217,orders,UNRECOGNIZED_DISCOUNT_CODE,0,


### Before the pipeline handoff

1. Verify all **nine source CSV names** match keys loaded in cell 1. Set `RAW_DIR` in Airflow; don't depend on notebook working directory.
2. Confirm the complete source-derived discount-code mapping (only `SAVE10` is shown as a **verified example**, not a complete mapping). Unrecognized codes stay unresolved, not zeroed.
3. Review the complete 12-category crosswalk and additional channel, province, courier, status, etc., mappings using `docs/dq_category_variants.csv`; source values are required before a safe complete dictionary can be supplied.
4. Resolve conflicting primary IDs, payment attempts, order-date anomalies and duplicate customer/product candidates before production transformations.
5. Commit this notebook and the decision log; port approved transformations into `include/clean/` as tested functions. This notebook is an **auditor and decision log**, not a silent production cleaner.

**Validation limitation:** Source CSVs weren't included in this upload, so a full real-data Restart & Run All is required before merging. The output counts may differ from the earlier report by design.